In [1]:
!pip install -q -U transformers datasets peft accelerate sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 16.8 MB/s eta 0:00:00


In [2]:
!pip uninstall -y torchao
!pip install -q -U "torchao>=0.16.0" peft

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 29.3 MB/s eta 0:00:00


In [3]:
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    DataCollatorForLanguageModeling
)

from datasets import load_dataset

from peft import (
    LoraConfig,
    get_peft_model
)

from torch.utils.data import DataLoader
from torch.optim import AdamW

In [4]:
print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

device = "cuda" if torch.cuda.is_available() else "cpu"

PyTorch version: 2.11.0+cu128
GPU available: True
GPU: Tesla T4


In [5]:
model_name = "distilgpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)

tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(model_name)

model = model.to(device)

print("Tokenizer loaded!")
print("Vocabulary size:", tokenizer.vocab_size)

print("Model loaded!")

total_params = sum(p.numel() for p in model.parameters())

print("Total parameters:", total_params)

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  353MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Tokenizer loaded!
Vocabulary size: 50257
Model loaded!
Total parameters: 81912576


In [6]:
text = "Machine learning is"

tokens = tokenizer.tokenize(text)

token_ids = tokenizer(text)["input_ids"]

print("\nText:", text)
print("Tokens:", tokens)
print("Token IDs:", token_ids)



Text: Machine learning is
Tokens: ['Machine', 'Ġlearning', 'Ġis']
Token IDs: [37573, 4673, 318]


In [7]:
dataset = load_dataset("Abirate/english_quotes")

print("\nDataset:")
print(dataset)

README.md:   0%|          | 0.00/5.55k [00:00<?, ?B/s]

quotes.jsonl: reconstructing file:   0%|          |  0.00B /  647kB            

quotes.jsonl: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/2508 [00:00<?, ? examples/s]


Dataset:
DatasetDict({
    train: Dataset({
        features: ['quote', 'author', 'tags'],
        num_rows: 2508
    })
})


In [8]:
train_dataset = dataset["train"].select(range(1000))

print("\nTraining samples:", len(train_dataset))


Training samples: 1000


In [9]:
def tokenize_function(example):

    return tokenizer(
        example["quote"],
        truncation=True,
        max_length=128,
        padding="max_length"
    )


tokenized_dataset = train_dataset.map(
    tokenize_function,
    batched=True
)

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [10]:
columns_to_remove = [
    column
    for column in tokenized_dataset.column_names
    if column not in ["input_ids", "attention_mask"]
]

tokenized_dataset = tokenized_dataset.remove_columns(
    columns_to_remove
)

print("\nTokenized dataset:")
print(tokenized_dataset)


Tokenized dataset:
Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 1000
})


In [11]:
def generate_text(
    prompt,
    max_new_tokens=50,
    temperature=0.8
):

    model.eval()

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():

        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    return tokenizer.decode(
        output[0],
        skip_special_tokens=True
    )

In [12]:
prompt = "Machine learning is"

result = generate_text(prompt)

print("\n" + "=" * 60)
print("BASE MODEL")
print("=" * 60)

print(result)


BASE MODEL
Machine learning is a major challenge at the moment, in part because, from a human perspective, learning can change all kinds of behaviour.




But the idea behind learning is to identify the factors that make learning good for all children. Even more so


In [13]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["c_attn"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

In [14]:
model = get_peft_model(
    model,
    lora_config
)

print("\nLoRA model:")

model.print_trainable_parameters()



LoRA model:
trainable params: 147,456 || all params: 82,060,032 || trainable%: 0.1797


/usr/local/lib/python3.13/dist-packages/peft/tuners/lora/layer.py:2631: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


In [15]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)


In [16]:
train_loader = DataLoader(
    tokenized_dataset,
    batch_size=4,
    shuffle=True,
    collate_fn=data_collator
)

print("\nNumber of batches:", len(train_loader))


Number of batches: 250


In [17]:
batch = next(iter(train_loader))

print("\nBatch shapes:")

for key, value in batch.items():

    print(
        key,
        value.shape,
        value.dtype
    )


Batch shapes:
input_ids torch.Size([4, 128]) torch.int64
attention_mask torch.Size([4, 128]) torch.int64
labels torch.Size([4, 128]) torch.int64


In [18]:
optimizer = AdamW(
    model.parameters(),
    lr=2e-4
)


In [19]:
epochs = 3

print("\n" + "=" * 60)
print("STARTING LoRA TRAINING")
print("=" * 60)

for epoch in range(epochs):

    model.train()

    total_loss = 0

    for step, batch in enumerate(train_loader):

        batch = {
            key: value.to(device)
            for key, value in batch.items()
        }

        # Forward pass
        outputs = model(**batch)

        loss = outputs.loss

        # Backward pass
        optimizer.zero_grad()

        loss.backward()

        # Update LoRA parameters
        optimizer.step()

        total_loss += loss.item()

        if (step + 1) % 50 == 0:

            print(
                f"Epoch {epoch + 1}/{epochs} | "
                f"Step {step + 1}/{len(train_loader)} | "
                f"Loss: {loss.item():.4f}"
            )

    avg_loss = total_loss / len(train_loader)

    print(
        f"Epoch {epoch + 1} completed | "
        f"Average Loss: {avg_loss:.4f}"
    )



STARTING LoRA TRAINING


[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch 1/3 | Step 50/250 | Loss: 3.2533
Epoch 1/3 | Step 100/250 | Loss: 3.9847
Epoch 1/3 | Step 150/250 | Loss: 3.2928
Epoch 1/3 | Step 200/250 | Loss: 3.3337
Epoch 1/3 | Step 250/250 | Loss: 3.1212
Epoch 1 completed | Average Loss: 3.5777
Epoch 2/3 | Step 50/250 | Loss: 3.4245
Epoch 2/3 | Step 100/250 | Loss: 3.7010
Epoch 2/3 | Step 150/250 | Loss: 3.5877
Epoch 2/3 | Step 200/250 | Loss: 3.3857
Epoch 2/3 | Step 250/250 | Loss: 3.0215
Epoch 2 completed | Average Loss: 3.4204
Epoch 3/3 | Step 50/250 | Loss: 3.2068
Epoch 3/3 | Step 100/250 | Loss: 3.2186
Epoch 3/3 | Step 150/250 | Loss: 3.1339
Epoch 3/3 | Step 200/250 | Loss: 3.5286
Epoch 3/3 | Step 250/250 | Loss: 3.9177
Epoch 3 completed | Average Loss: 3.3819


In [20]:
print("\nFinal LoRA parameters:")

model.print_trainable_parameters()



Final LoRA parameters:
trainable params: 147,456 || all params: 82,060,032 || trainable%: 0.1797


In [21]:
prompts = [
    "Machine learning is",
    "The future of technology",
    "Artificial intelligence can",
    "Education is important because"
]

print("\n" + "=" * 60)
print("FINE-TUNED LoRA MODEL")
print("=" * 60)

for prompt in prompts:

    print("\n" + "-" * 60)

    print("Prompt:", prompt)

    result = generate_text(
        prompt,
        max_new_tokens=50,
        temperature=0.8
    )

    print("Output:", result)


FINE-TUNED LoRA MODEL

------------------------------------------------------------
Prompt: Machine learning is
Output: Machine learning is that a new world of mind will be born upon us, and we will be able to imagine the moment of our own existence. In the world of consciousness, we will be born upon us, and we will be able to imagine the moment of our

------------------------------------------------------------
Prompt: The future of technology
Output: The future of technology is an open question. And we need to change the way we think about it.”—A. M. M. Maughan” The Future Of Technology is an open question. And we need to change the way we think about

------------------------------------------------------------
Prompt: Artificial intelligence can
Output: Artificial intelligence can be dangerous to our lives. But they can also be dangerous to our creativity and creativity.

------------------------------------------------------------
Prompt: Education is important because
Output: 